# NLP701 / NLP805 Assignment 1: Arabic Named Entity Recognition

This notebook trains a conditional random field (CRF) tagger on the ANERcorp
CamelLab split. It compares feature sets on a fixed development split, selects
the strongest development result, and produces the required test predictions.
The labels cover LOC, ORG, PER, and MISC entities in BIO format.


## Data preparation

The original training file is divided by sentence into training and development
sets with a fixed random seed. Test labels are not used during feature selection.
The dependency cell can be run when using a new Python environment.


In [ ]:
# Run once in a new Jupyter environment.
%pip install scikit-learn seqeval python-crfsuite


In [1]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

DATA_DIR = Path("ANERcorp-CamelLabSplits")
TRAIN_PATH = DATA_DIR / "ANERCorp_CamelLab_train.txt"
TEST_PATH = DATA_DIR / "ANERCorp_CamelLab_test.txt"

if not TRAIN_PATH.is_file() or not TEST_PATH.is_file():
    archive = Path("ANERcorp-CamelLabSplits.zip")
    if not archive.is_file():
        urlretrieve(
            "https://camel.abudhabi.nyu.edu/anercorp/ANERcorp-CamelLabSplits.zip",
            archive,
        )
    with ZipFile(archive) as zipped:
        zipped.extractall(".")

assert TRAIN_PATH.is_file() and TEST_PATH.is_file()


In [2]:
from sklearn.model_selection import train_test_split

def read_data(file_path):
    sentences = []
    sentence = []
    for line in Path(file_path).read_text(encoding="utf-8").splitlines():
        if line.strip():
            word, ner = line.split()
            sentence.append((word, ner.replace("PERS", "PER")))
        elif sentence:
            sentences.append(sentence)
            sentence = []
    if sentence:
        sentences.append(sentence)
    return sentences

all_train_sents = read_data(TRAIN_PATH)
train_sents, dev_sents = train_test_split(
    all_train_sents, test_size=0.1, random_state=12345
)
print(f"Training sentences: {len(train_sents)}; development sentences: {len(dev_sents)}")


Training sentences: 3575; development sentences: 398


## Feature engineering

`word2features(sent, i)` returns a feature dictionary for one token. The four
variants below share the original token and its immediate neighbors. Each row
adds to the preceding row:

| Variant | Additional features |
| --- | --- |
| `baseline` | Original token and immediate-neighbor words, plus sentence boundaries |
| `orthographic` | Normalized form, script shape, length, character-type flags, and neighbor forms |
| `affix` | Prefixes and suffixes of the token and immediate neighbors |
| `context2` | The preceding and following two-token context |

The feature function uses only token text and position; it does not read gold
labels when extracting features.


In [3]:
import re

ARABIC = re.compile(r"[\u0600-\u06ff]")
LATIN = re.compile(r"[A-Za-z]")
DIACRITICS = re.compile(r"[\u064b-\u065f\u0670\u0640]")
ALEF_MAP = str.maketrans({"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا", "ى": "ي"})
FINAL_VARIANT = "context2"

def normalize(word: str) -> str:
    return DIACRITICS.sub("", word).translate(ALEF_MAP)


def shape(word: str) -> str:
    codes = []
    for char in word:
        if char.isdigit():
            code = "D"
        elif ARABIC.fullmatch(char):
            code = "A"
        elif LATIN.fullmatch(char):
            code = "L"
        else:
            code = "P"
        if not codes or codes[-1] != code:
            codes.append(code)
    return "".join(codes)


def word2features(
    sent: list[tuple[str, str]], i: int, variant: str | None = None
) -> dict:
    if variant is None:
        variant = FINAL_VARIANT
    word = sent[i][0]
    features: dict = {"word": word}
    if i > 0:
        features["-1:word"] = sent[i - 1][0]
    else:
        features["BOS"] = True
    if i < len(sent) - 1:
        features["+1:word"] = sent[i + 1][0]
    else:
        features["EOS"] = True
    if variant == "baseline":
        return features

    norm = normalize(word)
    features.update(
        {
            "norm": norm,
            "shape": shape(word),
            "length_bucket": str(min(len(word), 12)),
            "has_digit": any(c.isdigit() for c in word),
            "has_latin": bool(LATIN.search(word)),
            "has_punct": any(not c.isalnum() for c in word),
            "starts_article": norm.startswith("ال"),
        }
    )
    for offset in (-1, 1):
        index = i + offset
        if 0 <= index < len(sent):
            other = sent[index][0]
            features[f"{offset:+}:norm"] = normalize(other)
            features[f"{offset:+}:shape"] = shape(other)
    if variant == "orthographic":
        return features

    for length in range(1, 5):
        if len(norm) >= length:
            features[f"prefix{length}"] = norm[:length]
            features[f"suffix{length}"] = norm[-length:]
    for offset in (-1, 1):
        index = i + offset
        if 0 <= index < len(sent):
            other = normalize(sent[index][0])
            for length in (2, 3):
                if len(other) >= length:
                    features[f"{offset:+}:prefix{length}"] = other[:length]
                    features[f"{offset:+}:suffix{length}"] = other[-length:]
    if variant == "affix":
        return features

    for offset in (-2, 2):
        index = i + offset
        if 0 <= index < len(sent):
            other = sent[index][0]
            features[f"{offset:+}:word"] = other
            features[f"{offset:+}:norm"] = normalize(other)
        else:
            features[f"{offset:+}:boundary"] = True
    if variant == "context2":
        return features
    raise ValueError(f"Unknown variant: {variant}")


def sent2features(
    sent: list[tuple[str, str]], variant: str | None = None
) -> list[dict]:
    return [word2features(sent, i, variant) for i in range(len(sent))]




## Development-set feature comparison

Every variant uses the same train/development split and CRF settings. The
selection criterion is entity-level F1 on the development set. Test data is
reserved for the final run.


In [4]:
import pycrfsuite
import tempfile
import time
from seqeval.metrics import (
    classification_report, f1_score, precision_score, recall_score
)

def sent2labels(sent):
    return [label for _, label in sent]

y_train = [sent2labels(sent) for sent in train_sents]
y_dev = [sent2labels(sent) for sent in dev_sents]
variants = ("baseline", "orthographic", "affix", "context2")
results = {}
model_paths = {}
model_tmpdir = tempfile.TemporaryDirectory(prefix="aner-crf-")

for variant in variants:
    X_train = [sent2features(sent, variant) for sent in train_sents]
    X_dev = [sent2features(sent, variant) for sent in dev_sents]

    trainer = pycrfsuite.Trainer(verbose=False)
    for xseq, yseq in zip(X_train, y_train):
        trainer.append(xseq, yseq)
    trainer.set_params({
        "c1": 1.0,
        "c2": 1e-3,
        "max_iterations": 50,
        "feature.possible_transitions": True,
    })

    model_path = Path(model_tmpdir.name) / f"{variant}.crfsuite"
    start = time.perf_counter()
    trainer.train(str(model_path))
    train_seconds = time.perf_counter() - start
    model_paths[variant] = model_path

    dev_tagger = pycrfsuite.Tagger()
    dev_tagger.open(str(model_path))
    dev_predictions = [dev_tagger.tag(xseq) for xseq in X_dev]
    results[variant] = {
        "precision": precision_score(y_dev, dev_predictions),
        "recall": recall_score(y_dev, dev_predictions),
        "f1": f1_score(y_dev, dev_predictions),
        "train_seconds": train_seconds,
    }
    metrics = results[variant]
    print(
        f"{variant:12s} P={metrics['precision']:.4f} "
        f"R={metrics['recall']:.4f} F1={metrics['f1']:.4f}"
    )

FINAL_VARIANT = max(variants, key=lambda variant: results[variant]["f1"])
print(f"Selected from development results: {FINAL_VARIANT}")


baseline     P=0.8980 R=0.5967 F1=0.7170
orthographic P=0.8746 R=0.6044 F1=0.7148
affix        P=0.8651 R=0.7267 F1=0.7899
context2     P=0.8692 R=0.7311 F1=0.7942
Selected from development results: context2


## Development-set evaluation

The table printed above compares all four variants. The classification report
below gives the selected variant's results by entity type.


In [5]:
final_tagger = pycrfsuite.Tagger()
final_tagger.open(str(model_paths[FINAL_VARIANT]))
X_dev = [sent2features(sent) for sent in dev_sents]
y_pred_dev = [final_tagger.tag(xseq) for xseq in X_dev]
print(classification_report(y_dev, y_pred_dev, digits=4))


              precision    recall  f1-score   support

         LOC     0.9429    0.8312    0.8835       397
        MISC     0.7910    0.6092    0.6883        87
         ORG     0.8120    0.6171    0.7013       175
         PER     0.8068    0.6929    0.7455       241

   micro avg     0.8692    0.7311    0.7942       900
   macro avg     0.8382    0.6876    0.7547       900
weighted avg     0.8663    0.7311    0.7923       900



## Final test evaluation and prediction export

This cell was executed once after selecting the feature set on the development
data. It writes `ANERCorp_CamelLab_test.pred.txt`, preserving token order and
sentence boundaries. The source data names the person entity `PERS`, so the
internal `PER` label is converted back when the prediction file is written.
The test result is not used to revise the model.


In [6]:
test_sents = read_data(TEST_PATH)
X_test = [sent2features(sent) for sent in test_sents]
y_pred_test = [final_tagger.tag(xseq) for xseq in X_test]

# This is the single final test-set evaluation.
y_test = [sent2labels(sent) for sent in test_sents]
print(f"Test entity-level F1: {f1_score(y_test, y_pred_test):.4f}")
print(classification_report(y_test, y_pred_test, digits=4))

prediction_blocks = []
for sent, predicted_labels in zip(test_sents, y_pred_test):
    assert len(sent) == len(predicted_labels)
    prediction_blocks.append("\n".join(
        f"{word} {label.replace('-PER', '-PERS')}"
        for (word, _), label in zip(sent, predicted_labels)
    ))
prediction_path = Path("ANERCorp_CamelLab_test.pred.txt")
prediction_path.write_text("\n\n".join(prediction_blocks), encoding="utf-8")

# Check token order and sentence separators without using test labels.
original_lines = TEST_PATH.read_text(encoding="utf-8").splitlines()
predicted_lines = prediction_path.read_text(encoding="utf-8").splitlines()
assert len(original_lines) == len(predicted_lines)
for original, predicted in zip(original_lines, predicted_lines):
    if not original:
        assert predicted == ""
    else:
        assert original.partition(" ")[0] == predicted.partition(" ")[0]
print(f"Saved {prediction_path} ({len(test_sents)} sentences)")


Test entity-level F1: 0.6117
              precision    recall  f1-score   support

         LOC     0.8185    0.7470    0.7811       676
        MISC     0.7597    0.4033    0.5269       243
         ORG     0.6888    0.3617    0.4743       459
         PER     0.7457    0.4333    0.5481       907

   micro avg     0.7675    0.5085    0.6117      2285
   macro avg     0.7532    0.4863    0.5826      2285
weighted avg     0.7573    0.5085    0.6000      2285

Saved ANERCorp_CamelLab_test.pred.txt (925 sentences)
